
# TSMOM 回测与分析一体化面板（成交级，对齐 IRP）

这份 notebook 将 **TSMOM 策略回测** 与 **固定权重基准对比** 放到与 IRP 基本一致的控制台结构里，重点做到：

1. 使用与 IRP 同一套成交级回测标准：  
   - `t` 日收盘生成信号  
   - `t+1` 按指定成交价成交  
   - 严格整手成交  
   - 区分估值价与交易价  
   - 支持成交额占比限制、停牌顺延、偏离触发调仓
2. 固定权重基准 **不复用策略过滤器**，避免过滤器污染 1:1 / 固定权重对比基准。
3. 图表和输出区块尽量与 IRP notebook 对齐，便于严格控制变量。

> 依赖：`market_data.py`、`backtest.py`、`backtest_utils.py`、`tsmom_utils.py`、`tsmom_backtest_aligned.py`


In [ ]:

from pathlib import Path

import seaborn as sns
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from IPython.display import display

from market_data import create_manager, today_str, load_tushare_token
from backtest import (
    build_market_matrices,
    calc_drawdown,
    performance_summary,
    calc_asset_correlation_matrix,
    calc_avg_pairwise_correlation,
)
from backtest_utils import (
    compare_fixed_weight_portfolios,
    summarize_relative_performance,
)
from tsmom_backtest import (
    make_tsmom_signal_snapshot,
    simulate_tsmom_backtest,
)

plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["axes.unicode_minus"] = False


In [ ]:

# ========= 资产池 =========
"""

WATCHLIST = [
    "510300.SH",  # 沪深300ETF
    "513100.SH",  # 纳斯达克100ETF
    "518880.SH",  # 黄金ETF
    "511010.SH",  # 国债ETF
    "162411.SZ",  # 华宝油气LOF
]
"""
WATCHLIST = [
    #"510300.SH",  # 沪深300ETF (核心权益：大盘蓝筹)
    #"515100.SH",  # 红利低波100ETF (抗通胀：红利股)
    "510880.SH",  # [替换] 红利ETF (华泰柏瑞) - 成立: 2006-11。完美平替 515100
    #"510500.SH",  # 中证500ETF (核心权益：中盘成长)
    #"159915.SZ",  # 创业板ETF (核心权益：高新成长 - 纯被动)
    
    "511090.SH",  # 30年国债ETF (避险资产：超长债，对利率极度敏感)
    #"511010.SH",  # 国债ETF (基础配置：中长期债)

    "518880.SH",  # 黄金ETF (抗通胀/避险：金属商品)
    "161226.SZ",  # 白银LOF
    #"510170.SH",  # 大宗商品50ETF (核心权益：大宗商品股票集合)
    

    #"159981.SZ",  # 能源ETF (抗通胀：能源/电力)


    "501018.SH",  # 南方原油LOF (抗通胀：国际原油价格)

    "159985.SZ",  # 豆粕ETF (抗通胀：农产品期货)
    #"159865.SZ",  # 养殖ETF （养殖类股票合集）

    #"513000.SH",  # 225日经ETF
    #"159659.SZ",  # 100纳斯达克ETF
    "513100.SH",  # 纳斯达克100ETF (国泰) - 成立: 2013-04。完美平替 159659，拥有极长历史的纳指标杆。
    
]

START_DATE = "20140801"
END_DATE = today_str()

# ========= 回测参数 =========
BACKTEST_PARAMS = {
    "initial_cash": 1_000_000.0,
    "lookback": [21, 63, 252],      #单周期写一个int，多周期用列表。
    "skip_recent": [0, 0, 21],       #和lookback需要等长。“算动量时，故意不看最近这几天。”
    "combination_method": "mean",    #用来定义多周期怎么合成一个总信号，常见可选："mean""weighted_mean""sum""median""vote"
    "horizon_weights": None,         #不填：默认等权，填了就按给的权重组合，比如 [0.2, 0.3, 0.5]，长度要和 lookback 一致
    "signal_type": "sign",           # sign / raw / rank / cross_sectional_zscore / time_series_zscore
    "use_excess_returns": True,
    "annual_rf": 0.02,
    "return_type": "simple",
    "vol_method": "ewma",            # ewma / rolling
    "vol_window": 63,
    "ewma_lambda": 0.94,
    "annualization": 252,
    "target_vol": 0.40,
    "max_abs_position": None,
    "rebalance_freq": "M",           # D / W / M / Q / Y
    "execution_price_type": "avg",   # open / close / high / low / avg
    "fee_rate_buy": 0.0005,
    "fee_rate_sell": 0.0005,
    "lot_size": 100,
    "max_trade_amount_ratio": 0.05,
    "amount_unit_scale": 1000.0,
    "use_drift_trigger": False,
    "drift_threshold": 0.05,
    "side": "long_only",             # 为了与 IRP 的 long-only 成交框架对齐，建议使用 long_only
    "signal_threshold": 0.0,
    "ma_filter_window": 120,         # None 表示不用 MA 过滤
    "normalize_to_gross": 1.0,
    "vol_floor": 1e-8,
    "normalize_trade_weights": True,
    "combination_method": "mean",   # 单周期可保留 mean；多周期示例：[63,126,252]
    "horizon_weights": None,       # None 表示等权组合各 lookback
    "signal_clip": 3.0,            # 对 zscore 类信号做截断
    "zero_to_nan": False,
    "risk_free_rate": 0.0,
    "valuation_ffill_limit": 5,
}

# 价格矩阵预处理参数
PRICE_PREPARE_KWARGS = {
    "ffill": True,
    "ffill_limit": 5,
    "min_non_na_ratio": 0.8,
    "drop_all_na_dates": True,
}

# 固定权重基准参数（与策略过滤器彻底解耦）
BASELINE_PARAMS = {
    "weights": None,                  # None 表示等权；也可传 dict
    "rebalance_freq": "M",
    "fee_rate": 0.0,
}

print("watchlist =", WATCHLIST)
print("params loaded")
display(pd.Series(BACKTEST_PARAMS, name="value").to_frame())
print("baseline params")
display(pd.Series(BASELINE_PARAMS, name="value").to_frame())


In [ ]:

# ========= 最近数据刷新（可选但推荐） =========
DO_REFRESH_RECENT = False
RECENT_LOOKBACK_DAYS = 7

TUSHARE_TOKEN = load_tushare_token()
DB_PATH = "data/db/market_data.db"
EXPORT_DIR = Path("data/exports_tsmom_aligned")
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

manager = create_manager(
    tushare_token=TUSHARE_TOKEN,
    db_path=DB_PATH,
    default_start_date="20150101",
    default_exchange="SSE",
)

print("manager ready")
print("db_path =", Path(DB_PATH).resolve())
print("export_dir =", EXPORT_DIR.resolve())

if DO_REFRESH_RECENT:
    recent_summary = manager.refresh_recent_daily_prices(
        ts_codes=WATCHLIST,
        lookback_days=RECENT_LOOKBACK_DAYS,
        end_date=END_DATE,
    )
    print("最近数据刷新结果：")
    display(pd.Series(recent_summary, name="value").to_frame())
else:
    print("已跳过最近数据刷新。")


In [ ]:

# ========= 读取市场数据并构建 market 字典（自动预加载 warmup） =========
def _as_int_list(x):
    if np.isscalar(x):
        return [int(x)]
    return [int(v) for v in x]

lookbacks = _as_int_list(BACKTEST_PARAMS["lookback"])
skip_recents = _as_int_list(BACKTEST_PARAMS["skip_recent"])
if len(skip_recents) == 1 and len(lookbacks) > 1:
    skip_recents = skip_recents * len(lookbacks)
elif len(skip_recents) != len(lookbacks):
    raise ValueError("当 lookback 为多周期时，skip_recent 需为标量或等长序列")

warmup_signal_bars = max(lb + sr for lb, sr in zip(lookbacks, skip_recents)) + 5
warmup_bars = max(
    warmup_signal_bars,
    int(BACKTEST_PARAMS["vol_window"]) + 5,
    int(BACKTEST_PARAMS["ma_filter_window"] or 0) + 5,
)
exchange = manager.config.default_exchange

open_trade_dates = pd.Index(manager.store.get_open_trade_dates(
    exchange=exchange,
    end_date=END_DATE,
)).astype(str)
open_trade_dates = open_trade_dates.sort_values()

visible_trade_dates = open_trade_dates[open_trade_dates >= START_DATE]
if len(visible_trade_dates) == 0:
    raise ValueError(f"数据库交易日历中找不到 START_DATE={START_DATE} 之后的交易日")

effective_start_date = str(visible_trade_dates[0])
trade_dates_up_to_start = open_trade_dates[open_trade_dates <= effective_start_date]

if len(trade_dates_up_to_start) == 0:
    data_start_date = START_DATE
else:
    preload_idx = max(0, len(trade_dates_up_to_start) - warmup_bars)
    data_start_date = str(trade_dates_up_to_start[preload_idx])

raw_prices = manager.store.get_daily_prices(
    ts_codes=WATCHLIST,
    start_date=data_start_date,
    end_date=END_DATE,
)

print("START_DATE =", START_DATE)
print("effective_start_date =", effective_start_date)
print("data_start_date (warmup) =", data_start_date)
print("warmup_bars =", warmup_bars)
print("raw_prices shape =", raw_prices.shape)
display(raw_prices.head())

coverage = raw_prices.groupby("ts_code")["trade_date"].agg(["min", "max", "count"])
print("资产覆盖情况：")
display(coverage)

market = build_market_matrices(
    data=raw_prices,
    fields=("open", "high", "low", "close", "amount"),
    date_col="trade_date",
    code_col="ts_code",
    date_format="%Y%m%d",
)

PRICE_PREPARE_KWARGS["calendar"] = market["close"].index

for k, v in market.items():
    print(k, v.shape)


In [ ]:

# ========= 运行 TSMOM 回测（先跑预热区间，再裁切回 START_DATE） =========
full_result = simulate_tsmom_backtest(
    market=market,
    initial_cash=BACKTEST_PARAMS["initial_cash"],
    lookback=BACKTEST_PARAMS["lookback"],
    skip_recent=BACKTEST_PARAMS["skip_recent"],
    signal_type=BACKTEST_PARAMS["signal_type"],
    use_excess_returns=BACKTEST_PARAMS["use_excess_returns"],
    annual_rf=BACKTEST_PARAMS["annual_rf"],
    return_type=BACKTEST_PARAMS["return_type"],
    vol_method=BACKTEST_PARAMS["vol_method"],
    vol_window=BACKTEST_PARAMS["vol_window"],
    ewma_lambda=BACKTEST_PARAMS["ewma_lambda"],
    annualization=BACKTEST_PARAMS["annualization"],
    target_vol=BACKTEST_PARAMS["target_vol"],
    max_abs_position=BACKTEST_PARAMS["max_abs_position"],
    rebalance_freq=BACKTEST_PARAMS["rebalance_freq"],
    execution_price_type=BACKTEST_PARAMS["execution_price_type"],
    valuation_ffill_limit=BACKTEST_PARAMS["valuation_ffill_limit"],
    fee_rate_buy=BACKTEST_PARAMS["fee_rate_buy"],
    fee_rate_sell=BACKTEST_PARAMS["fee_rate_sell"],
    lot_size=BACKTEST_PARAMS["lot_size"],
    max_trade_amount_ratio=BACKTEST_PARAMS["max_trade_amount_ratio"],
    amount_unit_scale=BACKTEST_PARAMS["amount_unit_scale"],
    use_drift_trigger=BACKTEST_PARAMS["use_drift_trigger"],
    drift_threshold=BACKTEST_PARAMS["drift_threshold"],
    price_prepare_kwargs=PRICE_PREPARE_KWARGS,
    side=BACKTEST_PARAMS["side"],
    signal_threshold=BACKTEST_PARAMS["signal_threshold"],
    ma_filter_window=BACKTEST_PARAMS["ma_filter_window"],
    normalize_to_gross=BACKTEST_PARAMS["normalize_to_gross"],
    vol_floor=BACKTEST_PARAMS["vol_floor"],
    normalize_trade_weights=BACKTEST_PARAMS["normalize_trade_weights"],
    combination_method=BACKTEST_PARAMS["combination_method"],
    horizon_weights=BACKTEST_PARAMS["horizon_weights"],
    signal_clip=BACKTEST_PARAMS["signal_clip"],
    zero_to_nan=BACKTEST_PARAMS["zero_to_nan"],
    risk_free_rate=BACKTEST_PARAMS["risk_free_rate"],
)

summary_full = full_result["summary"]
nav_df_full = full_result["nav_df"]
returns_full = full_result["returns"]
weights_df_full = full_result["weights_df"]
positions_df_full = full_result["positions_df"]
target_weights_df_full = full_result["target_weights_df"]
trades_df_full = full_result["trades_df"]
rebalance_log_df_full = full_result["rebalance_log_df"]
asset_corr_matrix_full = full_result["asset_corr_matrix"]
raw_signal_df_full = full_result["raw_signal_df"]
signal_df_full = full_result["signal_df"]
vol_df_full = full_result["vol_df"]
model_target_weights_df_full = full_result["model_target_weights_df"]
prepared_close_df_full = full_result["prepared_close_df"]

start_ts = pd.Timestamp(START_DATE)
visible_dates = nav_df_full.index[nav_df_full.index >= start_ts]
if len(visible_dates) == 0:
    raise ValueError(f"回测结果中不存在 {START_DATE} 之后的可见日期")

visible_start_ts = visible_dates[0]

nav_df = nav_df_full.loc[visible_start_ts:].copy()
weights_df = weights_df_full.loc[visible_start_ts:].copy()
positions_df = positions_df_full.loc[visible_start_ts:].copy()
target_weights_df = target_weights_df_full.loc[visible_start_ts:].copy()
raw_signal_df = raw_signal_df_full.loc[visible_start_ts:].copy()
signal_df = signal_df_full.loc[visible_start_ts:].copy()
vol_df = vol_df_full.loc[visible_start_ts:].copy()
model_target_weights_df = model_target_weights_df_full.loc[visible_start_ts:].copy()
prepared_close_df = prepared_close_df_full.loc[visible_start_ts:].copy()

trades_df = trades_df_full.copy()
if len(trades_df) > 0:
    trades_df["trade_date"] = pd.to_datetime(trades_df["trade_date"])
    trades_df = trades_df[trades_df["trade_date"] >= visible_start_ts].reset_index(drop=True)

rebalance_log_df = rebalance_log_df_full.copy()
if len(rebalance_log_df) > 0:
    rebalance_log_df["signal_date"] = pd.to_datetime(rebalance_log_df["signal_date"])
    rebalance_log_df["trade_date"] = pd.to_datetime(rebalance_log_df["trade_date"])
    rebalance_log_df = rebalance_log_df[rebalance_log_df["trade_date"] >= visible_start_ts].reset_index(drop=True)

if len(nav_df) > 0 and float(nav_df.iloc[0]["nav"]) > 0:
    display_scale = BACKTEST_PARAMS["initial_cash"] / float(nav_df.iloc[0]["nav"])
else:
    display_scale = 1.0

nav_df["nav"] = nav_df["nav"] * display_scale
nav_df["cash"] = nav_df["cash"] * display_scale
returns = nav_df["nav"].pct_change().fillna(0.0)

close_visible = market["close"].loc[visible_start_ts:].copy()
asset_corr_matrix = calc_asset_correlation_matrix(
    close_visible,
    return_type="log",
)
summary = performance_summary(
    nav=nav_df["nav"],
    returns=returns,
    risk_free_rate=BACKTEST_PARAMS["risk_free_rate"],
    annualization=BACKTEST_PARAMS["annualization"],
)
summary["avg_asset_correlation"] = calc_avg_pairwise_correlation(asset_corr_matrix)

print("TSMOM 成交级回测完成")
print("可见区间起点 =", visible_start_ts.strftime("%Y-%m-%d"))
print("说明：数据库已自动向前预加载 warmup 交易日用于热启动，展示与统计从 START_DATE 之后开始。")

display(summary.to_frame("value"))

print("关键参数快照：")
display(pd.DataFrame({
    "lookback": [str(BACKTEST_PARAMS["lookback"])],
    "skip_recent": [str(BACKTEST_PARAMS["skip_recent"])],
    "signal_type": [BACKTEST_PARAMS["signal_type"]],
    "combination_method": [BACKTEST_PARAMS["combination_method"]],
    "horizon_weights": [str(BACKTEST_PARAMS["horizon_weights"])],
    "rebalance_freq": [BACKTEST_PARAMS["rebalance_freq"]],
    "execution_price_type": [BACKTEST_PARAMS["execution_price_type"]],
    "warmup_data_start_date": [data_start_date],
    "visible_start_date": [visible_start_ts.strftime("%Y%m%d")],
    "display_scale": [display_scale],
}))


In [ ]:

# ========= 固定权重基准（与策略过滤器解耦） =========
benchmark_px = market["close"].loc[visible_start_ts:].copy().reindex(nav_df.index).ffill()

if BASELINE_PARAMS["weights"] is None:
    baseline_weight = pd.Series(1.0 / len(benchmark_px.columns), index=benchmark_px.columns, name="baseline_weight")
else:
    baseline_weight = pd.Series(BASELINE_PARAMS["weights"], dtype=float).reindex(benchmark_px.columns).fillna(0.0)
    baseline_weight = baseline_weight / baseline_weight.sum()
    baseline_weight.name = "baseline_weight"

baseline_compare = compare_fixed_weight_portfolios(
    price_df=benchmark_px,
    target_weights=baseline_weight,
    rebalance_freq=BASELINE_PARAMS["rebalance_freq"],
    initial_capital=BACKTEST_PARAMS["initial_cash"],
    fee_rate=BASELINE_PARAMS["fee_rate"],
    normalize=True,
    total_weight=1.0,
    drop_any_na=True,
)

buy_hold_nav = baseline_compare["buy_and_hold"]["nav"].rename("1to1_hold")
rebalanced_nav = baseline_compare["periodic_rebalance"]["nav"].rename(f"FixedWeight_{BASELINE_PARAMS['rebalance_freq']}")

buy_hold_returns = buy_hold_nav.pct_change().fillna(0.0)
rebalanced_returns = rebalanced_nav.pct_change().fillna(0.0)

buy_hold_summary = performance_summary(
    nav=buy_hold_nav,
    returns=buy_hold_returns,
    risk_free_rate=BACKTEST_PARAMS["risk_free_rate"],
    annualization=BACKTEST_PARAMS["annualization"],
)
rebalanced_summary = performance_summary(
    nav=rebalanced_nav,
    returns=rebalanced_returns,
    risk_free_rate=BACKTEST_PARAMS["risk_free_rate"],
    annualization=BACKTEST_PARAMS["annualization"],
)

same_pool_corr_matrix = calc_asset_correlation_matrix(
    benchmark_px,
    return_type="log",
)
avg_corr_same_pool = calc_avg_pairwise_correlation(same_pool_corr_matrix)
buy_hold_summary["avg_asset_correlation"] = avg_corr_same_pool
rebalanced_summary["avg_asset_correlation"] = avg_corr_same_pool

comparison_summary = pd.concat(
    [
        summary.rename("TSMOM"),
        buy_hold_summary.rename("1to1_hold"),
        rebalanced_summary.rename(f"FixedWeight_{BASELINE_PARAMS['rebalance_freq']}"),
    ],
    axis=1,
)

relative_to_buy_hold = summarize_relative_performance(
    strategy_nav=nav_df["nav"],
    benchmark_nav=buy_hold_nav,
    risk_free_annual=BACKTEST_PARAMS["risk_free_rate"],
    annualization=BACKTEST_PARAMS["annualization"],
)

relative_to_rebalanced = summarize_relative_performance(
    strategy_nav=nav_df["nav"],
    benchmark_nav=rebalanced_nav,
    risk_free_annual=BACKTEST_PARAMS["risk_free_rate"],
    annualization=BACKTEST_PARAMS["annualization"],
)

print("固定权重基准绩效：")
display(comparison_summary)
print("最近基准净值：")
display(pd.concat([buy_hold_nav, rebalanced_nav], axis=1).tail())


In [ ]:

# ========= 快速查看：资产净值 / 组合净值 / 回撤 / 现金占比 / 最近结果 =========
nav = nav_df["nav"].copy()
drawdown = calc_drawdown(nav)
cash_weight = nav_df["cash"] / nav_df["nav"]

asset_nav = benchmark_px.div(benchmark_px.iloc[0], axis=1)
equal_hold_drawdown = calc_drawdown(buy_hold_nav)
rebalanced_drawdown = calc_drawdown(rebalanced_nav)

fig, axes = plt.subplots(4, 1, figsize=(12, 13), sharex=True)

for col in asset_nav.columns:
    axes[0].plot(asset_nav.index, asset_nav[col], label=col, linewidth=1.2)
axes[0].set_title("Assets Normalized NAV (Start = 1)")
axes[0].legend(ncol=2, fontsize=9)

axes[1].plot(nav.index, nav / nav.iloc[0], label="TSMOM Portfolio NAV")
axes[1].plot(buy_hold_nav.index, buy_hold_nav / buy_hold_nav.iloc[0], label="1:1 Hold Benchmark", linestyle="--")
axes[1].plot(rebalanced_nav.index, rebalanced_nav / rebalanced_nav.iloc[0], label=f"FixedWeight_{BASELINE_PARAMS['rebalance_freq']}", linestyle=":")
axes[1].set_title("Portfolio NAV vs Fixed-Weight Benchmarks")
axes[1].legend()

axes[2].plot(drawdown.index, drawdown, label="TSMOM Drawdown")
axes[2].plot(equal_hold_drawdown.index, equal_hold_drawdown, label="1:1 Hold Drawdown", linestyle="--")
axes[2].plot(rebalanced_drawdown.index, rebalanced_drawdown, label=f"FixedWeight_{BASELINE_PARAMS['rebalance_freq']} Drawdown", linestyle=":")
axes[2].set_title("Drawdown Comparison")
axes[2].legend()

axes[3].plot(cash_weight.index, cash_weight, label="Cash Weight")
axes[3].set_title("Cash Weight")
axes[3].legend()

plt.tight_layout()
plt.show()

print("最近净值：")
display(nav_df.tail())

print("最近基准净值：")
display(pd.concat([buy_hold_nav.rename("1to1_hold"), rebalanced_nav], axis=1).tail())

print("最近现金占比：")
display(cash_weight.tail().to_frame("cash_weight"))

print("最近目标权重：")
display(target_weights_df.tail())

print("最近实际权重：")
display(weights_df.tail())

print("最近持仓：")
display(positions_df.tail())

print("最近调仓日志：")
display(rebalance_log_df.tail(10))

print("最近交易记录：")
display(trades_df.tail(20))

print("平均现金占比：", cash_weight.mean())
print("最大现金占比：", cash_weight.max())
print("TSMOM 最大回撤：", drawdown.min())
print("1:1 最大回撤：", equal_hold_drawdown.min())


In [ ]:

# ========= 资产相关性矩阵 + 实际持仓权重变化 =========
print("资产相关性矩阵：")
display(asset_corr_matrix)

plt.figure(figsize=(8, 6))
sns.heatmap(
    asset_corr_matrix,
    annot=True,
    cmap="coolwarm",
    vmin=-1,
    vmax=1,
    center=0,
    fmt=".2f",
    linewidths=1,
    square=True,
)
plt.title("Asset Correlation Matrix", fontsize=14, pad=15)
plt.tight_layout()
plt.show()

actual_weights_plot = weights_df.copy().fillna(0.0)

plt.figure(figsize=(14, 6))
if float(actual_weights_plot.min().min()) >= -1e-12:
    plt.stackplot(
        actual_weights_plot.index,
        actual_weights_plot.T.values,
        labels=actual_weights_plot.columns,
    )
else:
    actual_weights_plot.plot.line(ax=plt.gca(), linewidth=1.2)
plt.title("Actual Portfolio Weights Over Time")
plt.xlabel("Date")
plt.ylabel("Weight")
plt.legend(loc="upper left", bbox_to_anchor=(1.01, 1.0))
plt.tight_layout()
plt.show()


In [ ]:

# ========= 历史目标权重 + 基础序列复核 =========
fig, ax = plt.subplots(figsize=(12, 6))

target_weights_df.plot.line(
    ax=ax,
    linewidth=1.5,
    alpha=0.8,
    colormap="Set1",
)

plt.axhline(0, color="black", linewidth=1.5, linestyle="-", zorder=1)

plt.title("Historical TSMOM Target Weights", fontsize=14, pad=15)
plt.xlabel("Date", fontsize=12)
plt.ylabel("Target Weight", fontsize=12)

handles, labels = ax.get_legend_handles_labels()
plt.legend(handles, labels, loc="upper left", bbox_to_anchor=(1.02, 1))
plt.tight_layout()
plt.show()

print("平均现金占比：", round(cash_weight.mean(), 4))
print("最大现金占比：", round(cash_weight.max(), 4))
print("最近现金占比：", round(cash_weight.iloc[-1], 4))
print("最近净值：", round(nav.iloc[-1], 4))
print("样本期最大回撤：", round(drawdown.min(), 4))


In [ ]:

# ========= 年度收益 + 月度收益热力图 =========
year_end_nav = nav.resample("Y").last()
annual_returns = year_end_nav.pct_change()
annual_returns.index = annual_returns.index.year

annual_returns_df = annual_returns.to_frame("annual_return")
print("年度收益：")
display(annual_returns_df)

plt.figure(figsize=(10, 4))
plt.bar(annual_returns_df.index.astype(str), annual_returns_df["annual_return"])
plt.title("Annual Returns")
plt.show()

month_end_nav = nav.resample("M").last()
monthly_returns = month_end_nav.pct_change()
monthly_df = monthly_returns.to_frame("monthly_return")
monthly_df["year"] = monthly_df.index.year
monthly_df["month"] = monthly_df.index.month

heatmap_table = monthly_df.pivot(index="year", columns="month", values="monthly_return")
print("月度收益热力图（表格版）：")
display(heatmap_table.style.format("{:.2%}"))

fig, ax = plt.subplots(figsize=(12, 6))
im = ax.imshow(heatmap_table.values, aspect="auto")

ax.set_xticks(range(len(heatmap_table.columns)))
ax.set_xticklabels(heatmap_table.columns)

ax.set_yticks(range(len(heatmap_table.index)))
ax.set_yticklabels(heatmap_table.index)

ax.set_title("Monthly Return Heatmap")
ax.set_xlabel("Month")
ax.set_ylabel("Year")

for i in range(heatmap_table.shape[0]):
    for j in range(heatmap_table.shape[1]):
        val = heatmap_table.iloc[i, j]
        if pd.notna(val):
            ax.text(j, i, f"{val:.1%}", ha="center", va="center", fontsize=9)

fig.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()


In [ ]:

# ========= 调仓统计 + 交易统计 =========
if len(rebalance_log_df) > 0:
    rebalance_log_df["trade_date"] = pd.to_datetime(rebalance_log_df["trade_date"])
    rebalance_log_df["year"] = rebalance_log_df["trade_date"].dt.year

    print("调仓日志：")
    display(rebalance_log_df.tail(20))

    yearly_rebalance_count = rebalance_log_df.groupby("year").size().rename("rebalance_count")
    yearly_turnover = rebalance_log_df.groupby("year")["turnover"].sum().rename("turnover_sum")

    print("按年调仓统计：")
    display(pd.concat([yearly_rebalance_count, yearly_turnover], axis=1))
else:
    print("没有调仓记录")

if len(trades_df) > 0:
    trade_summary = trades_df.groupby(["ts_code", "side"])["trade_value"].agg(["count", "sum"])
    fee_summary = trades_df.groupby("side")["cost"].sum().rename("total_cost")

    print("交易统计：")
    display(trade_summary)

    print("手续费统计：")
    display(fee_summary)
else:
    print("没有交易记录")


In [ ]:

# ========= 权重分析 + 最近持仓与目标权重对比 =========
print("绩效摘要：")
display(comparison_summary)

print("相对 1:1 持仓：")
display(relative_to_buy_hold.to_frame("value"))

print("相对固定权重周期再平衡：")
display(relative_to_rebalanced.to_frame("value"))

print("策略权重统计：")
display(full_result["weight_stats"])

print("策略暴露统计：")
display(full_result["exposure_stats"])

print("最后一期实际权重：")
if len(weights_df) > 0:
    display(
        weights_df.tail(1).T.rename(columns={weights_df.index[-1]: "last_weight"})
        .sort_values("last_weight", ascending=False)
    )

print("样本期平均实际权重：")
if len(weights_df) > 0:
    avg_weight = weights_df.mean().sort_values(ascending=False).to_frame("avg_weight")
    display(avg_weight)

    plt.figure(figsize=(10, 4))
    avg_weight["avg_weight"].plot(kind="bar")
    plt.title("Average Actual Weights")
    plt.show()

print("最近持仓与目标权重对比：")
if len(weights_df) > 0 and len(target_weights_df) > 0:
    last_actual = weights_df.tail(1).T.rename(columns={weights_df.index[-1]: "actual_weight"})
    last_target = target_weights_df.tail(1).T.rename(columns={target_weights_df.index[-1]: "target_weight"})
    compare = last_actual.join(last_target, how="outer").fillna(0.0)
    compare["diff"] = compare["actual_weight"] - compare["target_weight"]
    display(compare.sort_values("target_weight", ascending=False))

print("最近持仓股数：")
if len(positions_df) > 0:
    display(
        positions_df.tail(1).T.rename(columns={positions_df.index[-1]: "last_position"})
        .sort_values("last_position", ascending=False)
    )

snapshot = make_tsmom_signal_snapshot(
    close_price_df=market["close"],
    signal_date=nav_df.index[-1],
    price_prepare_kwargs=PRICE_PREPARE_KWARGS,
    lookback=BACKTEST_PARAMS["lookback"],
    skip_recent=BACKTEST_PARAMS["skip_recent"],
    signal_type=BACKTEST_PARAMS["signal_type"],
    use_excess_returns=BACKTEST_PARAMS["use_excess_returns"],
    annual_rf=BACKTEST_PARAMS["annual_rf"],
    return_type=BACKTEST_PARAMS["return_type"],
    vol_method=BACKTEST_PARAMS["vol_method"],
    vol_window=BACKTEST_PARAMS["vol_window"],
    ewma_lambda=BACKTEST_PARAMS["ewma_lambda"],
    annualization=BACKTEST_PARAMS["annualization"],
    target_vol=BACKTEST_PARAMS["target_vol"],
    max_abs_position=BACKTEST_PARAMS["max_abs_position"],
    side=BACKTEST_PARAMS["side"],
    signal_threshold=BACKTEST_PARAMS["signal_threshold"],
    ma_filter_window=BACKTEST_PARAMS["ma_filter_window"],
    normalize_to_gross=BACKTEST_PARAMS["normalize_to_gross"],
    vol_floor=BACKTEST_PARAMS["vol_floor"],
    normalize_trade_weights=BACKTEST_PARAMS["normalize_trade_weights"],
    combination_method=BACKTEST_PARAMS["combination_method"],
    horizon_weights=BACKTEST_PARAMS["horizon_weights"],
    signal_clip=BACKTEST_PARAMS["signal_clip"],
    zero_to_nan=BACKTEST_PARAMS["zero_to_nan"],
)

snapshot_df = pd.concat(
    [snapshot.raw_signal, snapshot.signal, snapshot.volatility, snapshot.target_weight],
    axis=1,
).sort_values("weight", ascending=False)

print("最近一个 signal_date 的信号快照：", snapshot.signal_date)
display(snapshot_df)


In [ ]:

# ========= 一键导出 =========
nav_df.to_csv(EXPORT_DIR / "nav.csv")
weights_df.to_csv(EXPORT_DIR / "weights.csv")
positions_df.to_csv(EXPORT_DIR / "positions.csv")
target_weights_df.to_csv(EXPORT_DIR / "target_weights.csv")
trades_df.to_csv(EXPORT_DIR / "trades.csv", index=False)
rebalance_log_df.to_csv(EXPORT_DIR / "rebalance_log.csv", index=False)
summary.to_csv(EXPORT_DIR / "summary.csv")

asset_corr_matrix.to_csv(EXPORT_DIR / "asset_corr_matrix.csv")
raw_signal_df.to_csv(EXPORT_DIR / "raw_signal_df.csv")
signal_df.to_csv(EXPORT_DIR / "signal_df.csv")
vol_df.to_csv(EXPORT_DIR / "vol_df.csv")
model_target_weights_df.to_csv(EXPORT_DIR / "model_target_weights_df.csv")
comparison_summary.to_csv(EXPORT_DIR / "comparison_summary.csv")
relative_to_buy_hold.to_frame("value").to_csv(EXPORT_DIR / "relative_to_buy_hold.csv")
relative_to_rebalanced.to_frame("value").to_csv(EXPORT_DIR / "relative_to_rebalanced.csv")
pd.concat([buy_hold_nav.rename("1to1_hold"), rebalanced_nav], axis=1).to_csv(EXPORT_DIR / "baseline_navs.csv")

run_meta = pd.Series({
    "start_date": START_DATE,
    "end_date": END_DATE,
    "warmup_data_start_date": data_start_date,
    "visible_start_date": visible_start_ts.strftime("%Y%m%d"),
    "display_scale": display_scale,
})
run_meta.to_csv(EXPORT_DIR / "run_meta.csv")

print("export done")
display(summary.to_frame("value"))
display(run_meta.to_frame("value"))


In [ ]:

# ========= 可选：快速改参数重跑 =========
# 你可以直接修改 BACKTEST_PARAMS / BASELINE_PARAMS / WATCHLIST，
# 然后从“读取市场数据并构建 market 字典”开始重新运行。
#
# 若只改图表展示，可以从后半段对应 cell 重新运行；
# 若改了信号参数 / 调仓参数 / 资产池，建议从回测主入口开始重跑。
